In [23]:
import requests
import os 
import sys
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Optional, Dict
url =  "https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce"

root_dir = Path.cwd()
downloads_dir = root_dir / "data"

def download_dataset(dest_dir, file_name):
    dest_dir.mkdir(parents=True, exist_ok=True) 
    output_path = dest_dir / file_name
    
    if os.path.exists(output_path):
        print(f"dataset already exists at {output_path}")
        return
        
    try:
        with requests.get(url, stream=True, allow_redirects=True) as r:
            r.raise_for_status()  
    
            total_size = int(r.headers.get("content-length", 0))
            block_size = 8192 
    
            with open(output_path, "wb") as f:
                downloaded_size = 0
                for chunk in r.iter_content(chunk_size=block_size):
                    if chunk:
                        f.write(chunk)
                        downloaded_size += len(chunk)
                        progress_percent = (downloaded_size / total_size) * 100 if total_size else 0
                        print(f"\rDownloaed ({progress_percent:.2f}%)", end="")
                        
            print(f"\nSuccessfully downloaded '{file_name}' to '{output_path}'.")
    except requests.exceptions.RequestException as e:
        print(f"Error during download: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")


download_dataset(downloads_dir, file_name="brazilian-ecommerce.zip")


dataset already exists at /adev/repo/python/ml-study/data-analysis/data/brazilian-ecommerce.zip


## Unzip the dataset

In [36]:


def unzip_dataset(file_path, contents: List[str]|None = None)->Dict[str, pd.DataFrame]:
    dataframes = {}
    try:
        with zipfile.ZipFile(file_path,'r') as zf:
            zip_contents = zf.namelist()
            contents = contents if contents else zip_contents
            for content in contents:
                if content in zip_contents:
                    with zf.open(content) as f_zf:
                       dataframes[content] = pd.read_csv(f_zf) 
        return dataframes
    except zipfile.BadZipFile:
        print(f"Erro: {file_path} not valid ZIP file or corrupted")
    except Exception as e:
        print(f"An error occurred: {e}")

def dataset_info_from_zip(file_path) -> str:
    with zipfile.ZipFile(file_path, 'r') as zf:
        return zf.namelist()
    
        


In [37]:
dataset_info_from_zip(output_path)

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [38]:
dfs_dict = unzip_dataset(output_path)

dict_keys(['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv'])
